# 08 - External Validation (Late Fusion Architecture)

This notebook demonstrates external validation with Late Fusion:

**KEY FEATURE:** Handles missing clinical data by falling back to Genomic Model only

In [1]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import joblib
import config
from src.io import save_table, logger
from src.models import xgb_safe_frame
from src.fusion.late_fusion import LateFusionPredictor, fallback_to_genomic_only
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from src.visualization import setup_style
setup_style()

## Step 1: Load Trained Models

In [2]:
genomic_model = joblib.load(config.MODELS_DIR / "genomic_model.joblib")
clinical_model = joblib.load(config.MODELS_DIR / "clinical_model.joblib")
fusion_predictor = joblib.load(config.MODELS_DIR / "late_fusion_predictor.joblib")

print("Loaded models:")
print(f"  - genomic_model.joblib")
print(f"  - clinical_model.joblib")
print(f"  - late_fusion_predictor.joblib")
print(f"\nFusion weights: genomic={fusion_predictor.genomic_weight:.3f}, clinical={fusion_predictor.clinical_weight:.3f}")

Loaded models:
  - genomic_model.joblib
  - clinical_model.joblib
  - late_fusion_predictor.joblib

Fusion weights: genomic=0.610, clinical=0.390


## Step 2: Load External Validation Data

In [3]:
external_dir = config.DATA_DIR / "external"
external_dir.mkdir(parents=True, exist_ok=True)

try:
    X_ext = pd.read_csv(external_dir / "X_external.csv")
    y_ext = pd.read_csv(external_dir / "y_external.csv").iloc[:, 0]
    print(f"External data shape: {X_ext.shape}, positives: {int(y_ext.sum())}")
except FileNotFoundError:
    print("No external data found. Using internal test set for demonstration.")
    X_ext = pd.read_csv(config.PROCESSED_DIR / "X_test_preprocessed.csv")
    y_ext = pd.read_csv(config.PROCESSED_DIR / "y_test.csv").iloc[:, 0]
    print(f"Using internal test set: {X_ext.shape}, positives: {int(y_ext.sum())}")

No external data found. Using internal test set for demonstration.
Using internal test set: (86, 19019), positives: 12


## Step 3: Prepare Features for External Data

In [4]:
from src.genomic_selector import transform_genomic
from src.clinical_engineer import create_clinical_features

genomic_features = pd.read_csv(config.TABLES_DIR / "final_genomic_features.csv")["feature"].tolist()
clinical_features = pd.read_csv(config.TABLES_DIR / "final_clinical_features.csv")["feature"].tolist()

GENE_COLS = [c for c in X_ext.columns if c not in [
    'Gleason pattern primary', 'Gleason pattern secondary',
    'Surgical Margin Resection Status_R1',
    'Primary Lymph Node Presentation Assessment Ind-3_YES'
] and not any(x in c for x in ['Tumor Stage Code_', 'pathology'])]

CLINICAL_COLS = [c for c in X_ext.columns if c not in GENE_COLS]

X_ext_genomic = X_ext[GENE_COLS]
X_ext_clinical = X_ext[CLINICAL_COLS]

X_ext_genomic_selected = transform_genomic(X_ext_genomic, 
    {"imputer": genomic_model.named_steps['preprocessor'].named_steps['imputer'] if hasattr(genomic_model, 'named_steps') else None},
    genomic_features
) if hasattr(genomic_model, 'named_steps') else X_ext_genomic[genomic_features]

X_ext_clinical_eng, _ = create_clinical_features(X_ext_clinical)
available_clinical = [f for f in clinical_features if f in X_ext_clinical_eng.columns]
X_ext_clinical_final = X_ext_clinical_eng[available_clinical] if available_clinical else None

has_full_clinical = X_ext_clinical_final is not None and len(available_clinical) == len(clinical_features)
print(f"Has full clinical data: {has_full_clinical}")
print(f"Available clinical features: {len(available_clinical)}/{len(clinical_features)}")

2026-09-11 16:27:51 | INFO     | prostate_bcr | Clinical: Gleason_Total, High_Risk_Gleason
2026-09-11 16:27:51 | INFO     | prostate_bcr | Clinical: Margin_x_LymphNode
2026-09-11 16:27:51 | INFO     | prostate_bcr | Clinical: T_Stage_Risk


Has full clinical data: True
Available clinical features: 14/14


## Step 4: Predict with Fallback Logic

In [5]:
if has_full_clinical:
    proba = fusion_predictor.predict_proba(X_ext_genomic_selected, X_ext_clinical_final)[:, 1]
    predictions = (proba >= 0.5).astype(int)
    print("Using full Late Fusion (genomic + clinical)")
else:
    proba = fallback_to_genomic_only(genomic_model, X_ext_genomic_selected)
    proba = proba[:, 1] if len(proba.shape) > 1 else proba
    predictions = (proba >= 0.5).astype(int)
    print("FALLBACK: Using genomic model only (clinical data incomplete)")

auc = roc_auc_score(y_ext, proba)
accuracy = accuracy_score(y_ext, predictions)

print(f"\n=== External Validation Results ===")
print(f"AUC:       {auc:.4f}")
print(f"Accuracy:  {accuracy:.4f}")
print(f"\nClassification Report:\n{classification_report(y_ext, predictions)}")

Using full Late Fusion (genomic + clinical)

=== External Validation Results ===
AUC:       0.8401
Accuracy:  0.8488

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.93      0.91        74
           1       0.44      0.33      0.38        12

    accuracy                           0.85        86
   macro avg       0.67      0.63      0.65        86
weighted avg       0.83      0.85      0.84        86



## Step 5: Compare Individual Branch Performance

In [6]:
proba_genomic = genomic_model.predict_proba(X_ext_genomic_selected)[:, 1]
auc_genomic = roc_auc_score(y_ext, proba_genomic)

if has_full_clinical:
    proba_clinical = clinical_model.predict_proba(X_ext_clinical_final)[:, 1]
    auc_clinical = roc_auc_score(y_ext, proba_clinical)
else:
    auc_clinical = np.nan

print("Branch Performance Comparison:")
print(f"  Genomic AUC:  {auc_genomic:.4f}")
print(f"  Clinical AUC: {auc_clinical:.4f}" if not np.isnan(auc_clinical) else "  Clinical AUC: N/A (missing data)")
print(f"  Fusion AUC:   {auc:.4f}")

Branch Performance Comparison:
  Genomic AUC:  0.7579
  Clinical AUC: 0.7832
  Fusion AUC:   0.8401


## Step 6: Save Predictions

In [7]:
results_df = pd.DataFrame({
    "sample_id": range(len(y_ext)),
    "true_label": y_ext.values,
    "predicted_proba": proba,
    "predicted_label": predictions,
    "method": "late_fusion" if has_full_clinical else "genomic_only",
})

results_df.to_csv(config.TABLES_DIR / "external_validation_results.csv", index=False)
print(f"\nSaved predictions to {config.TABLES_DIR / 'external_validation_results.csv'}")


Saved predictions to D:\Prostate_BCR\core\outputs\tables\external_validation_results.csv
